# Silver Layer — Clean & Enrich Hospitality Data
Deduplicates bronze tables, parses dates, derives ADR, length of stay, occupancy inputs, and satisfaction scores.

In [ ]:
from pyspark.sql.functions import (
    col, to_date, datediff, round as spark_round, when, lit,
    coalesce, trim, upper, avg, current_timestamp
)

In [ ]:
# Silver properties — clean and standardise
df_props = spark.read.format('delta').table('bronze_properties')
df_props = (
    df_props
    .dropDuplicates(['property_id'])
    .withColumn('property_name', trim(col('property_name')))
    .withColumn('city',          trim(col('city')))
    .withColumn('country',       upper(trim(col('country'))))
    .withColumn('silver_timestamp', current_timestamp())
)
df_props.write.mode('overwrite').format('delta').saveAsTable('silver_properties')
print(f'Silver properties: {spark.table("silver_properties").count()} rows')

In [ ]:
# Silver guests — clean loyalty tier, standardise region
df_guests = spark.read.format('delta').table('bronze_guests')
df_guests = (
    df_guests
    .dropDuplicates(['guest_id'])
    .withColumn('loyalty_tier', trim(col('loyalty_tier')))
    .withColumn('region',       trim(col('region')))
    .withColumn('signup_date',  to_date(col('signup_date'), 'yyyy-MM-dd'))
    .withColumn('silver_timestamp', current_timestamp())
)
df_guests.write.mode('overwrite').format('delta').saveAsTable('silver_guests')
print(f'Silver guests: {spark.table("silver_guests").count()} rows')

In [ ]:
# Silver bookings — parse dates, derive length_of_stay, adr_band, revenue_flag
df_bk = spark.read.format('delta').table('bronze_bookings')
df_bk = (
    df_bk
    .dropDuplicates(['booking_id'])
    .withColumn('check_in_date',  to_date(col('check_in_date'),  'yyyy-MM-dd'))
    .withColumn('check_out_date', to_date(col('check_out_date'), 'yyyy-MM-dd'))
    .withColumn('length_of_stay', datediff(col('check_out_date'), col('check_in_date')))
    .withColumn('adr_band',
        when(col('room_rate') < 100, 'Budget')
        .when(col('room_rate') < 200, 'Mid-Scale')
        .when(col('room_rate') < 400, 'Upscale')
        .otherwise('Luxury')
    )
    .withColumn('is_completed',
        when(col('status') == 'completed', 1).otherwise(0)
    )
    .withColumn('is_cancelled',
        when(col('status') == 'cancelled', 1).otherwise(0)
    )
    .withColumn('revenue',
        when(col('status') == 'completed', col('total_amount')).otherwise(lit(0.0))
    )
    .withColumn('silver_timestamp', current_timestamp())
)
df_bk.write.mode('overwrite').format('delta').saveAsTable('silver_bookings')
print(f'Silver bookings: {spark.table("silver_bookings").count()} rows')

In [ ]:
# Silver reviews — parse date, derive satisfaction_band
df_rev = spark.read.format('delta').table('bronze_reviews')
df_rev = (
    df_rev
    .dropDuplicates(['review_id'])
    .withColumn('review_date', to_date(col('review_date'), 'yyyy-MM-dd'))
    .withColumn('composite_score',
        spark_round(
            (col('cleanliness_score') + col('service_score') +
             col('value_score') + col('food_score')) / 4.0,
            2
        )
    )
    .withColumn('satisfaction_band',
        when(col('overall_score') >= 9, 'Excellent')
        .when(col('overall_score') >= 7, 'Good')
        .when(col('overall_score') >= 5, 'Fair')
        .otherwise('Poor')
    )
    .withColumn('silver_timestamp', current_timestamp())
)
df_rev.write.mode('overwrite').format('delta').saveAsTable('silver_reviews')
print(f'Silver reviews: {spark.table("silver_reviews").count()} rows')

In [ ]:
print('Silver transformation complete.')
print(f'  Properties : {spark.read.format("delta").table("silver_properties").count():>7,}')
print(f'  Guests     : {spark.read.format("delta").table("silver_guests").count():>7,}')
print(f'  Bookings   : {spark.read.format("delta").table("silver_bookings").count():>7,}')
print(f'  Reviews    : {spark.read.format("delta").table("silver_reviews").count():>7,}')